# SteerMoE expert detection — vector construction

Replicates the detection phase of **"Steering MoE LLMs via Expert (De)Activation"**
([arXiv:2509.09660](https://arxiv.org/abs/2509.09660),
[official code](https://github.com/adobe-research/SteerMoE)) — the official
`custom_steering.ipynb` demo — with EasySteer's `router_logits` capture
stream, no forked model code needed:

1. Capture per-token router logits for a few **contrastive pairs**
   (answering with digits `1, 2, 3` vs. words `one, two, three`).
2. Compute each expert's top-k selection rate on the behavior tokens and
   the **risk difference** `Δ = p_digits − p_words`.
3. Save the top digit-linked experts as a `deactivate` steering config for
   `steermoe_steer.ipynb`.

Model: `allenai/OLMoE-1B-7B-0125-Instruct` (16 MoE layers × 64 experts,
top-8), one of the six models evaluated in the paper.

In [1]:
import json
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
from vllm import LLM, SamplingParams
from vllm.hidden_states import deserialize_hidden_states

MODEL = os.path.expanduser("~/models/OLMoE-1B-7B-0125-Instruct")  # allenai/OLMoE-1B-7B-0125-Instruct

with open(os.path.join(MODEL, "config.json")) as f:
    hf_cfg = json.load(f)
N_EXPERTS = hf_cfg["num_experts"]      # 64
TOP_K = hf_cfg["num_experts_per_tok"]  # 8

# Router-logit capture uses gate forward hooks, so the engine must run
# eagerly; prefix caching stays off because cache-hit tokens are never
# recomputed and so could never be captured.
llm = LLM(
    model=MODEL,
    enforce_eager=True,
    tensor_parallel_size=1,
    enable_chunked_prefill=False,
    enable_prefix_caching=False,
    gpu_memory_utilization=0.4,
    max_model_len=4096,
)
tok = llm.get_tokenizer()


def rpc(method, *args, **kwargs):
    return llm.llm_engine.collective_rpc(method, args=args, kwargs=kwargs)[0]

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-03 05:05:24 [api_utils.py:273] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': '/home/xhl/models/OLMoE-1B-7B-0125-Instruct'}


INFO 08-03 05:05:24 [model.py:623] Resolved architecture: OlmoeForCausalLM


INFO 08-03 05:05:24 [model.py:1788] Using max model len 4096


...

WARNING 08-03 05:05:58 [vllm.py:1258] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=2393275) 

INFO 08-03 05:05:58 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


(EngineCore pid=2393275) 

INFO 08-03 05:05:58 [vllm.py:1437] Cudagraph is disabled under eager mode


## 1. The contrastive pairs

Each side renders a full chat turn **including the assistant response**, so
a single prefill routes every response token through the MoE layers. The
`target` string marks the tokens whose routings we compare. The official
demo uses a single pair; on a small 64-expert model a few pairs sharpen the
risk difference considerably.

In [2]:
PAIRS = [
    ("Count to ten",
     "1, 2, 3, 4, 5, 6, 7, 8, 9, 10",
     "one, two, three, four, five, six, seven, eight, nine, ten"),
    ("How many days are in a week, and how many months in a year?",
     "There are 7 days in a week and 12 months in a year.",
     "There are seven days in a week and twelve months in a year."),
    ("What is five plus three?",
     "5 + 3 = 8",
     "five plus three equals eight"),
]

## 2. Capture router logits

`start_capture("router_logits")` just turns the stream on — the capture
hooks already sit on every MoE gate. One prefill later, `fetch_captured`
returns `{layer: (num_tokens, n_experts)}`.

In [3]:
def find_sub_list(sub, seq):
    n = len(sub)
    return [(i, i + n - 1) for i in range(len(seq) - n + 1)
            if seq[i:i + n] == sub]


def topk_membership(rows):
    """(tokens, n_experts) logits -> bool top-k membership mask."""
    order = np.argsort(rows, axis=-1)[:, -TOP_K:]
    mask = np.zeros(rows.shape, dtype=bool)
    np.put_along_axis(mask, order, True, axis=-1)
    return mask


counts = {"digits": None, "words": None}
totals = {"digits": 0, "words": 0}
for user, digits_ans, words_ans in PAIRS:
    for key, answer in (("digits", digits_ans), ("words", words_ans)):
        msgs = [{"role": "user", "content": user},
                {"role": "assistant", "content": answer}]
        text = tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=False)
        prompt_ids = tok(text, add_special_tokens=False).input_ids

        rpc("start_capture", "router_logits")
        llm.generate({"prompt_token_ids": prompt_ids},
                     sampling_params=SamplingParams(temperature=0.0,
                                                    max_tokens=1))
        logits = {lid: t.float().numpy()
                  for lid, t in deserialize_hidden_states(
                      rpc("fetch_captured", "router_logits")).items()}
        rpc("stop_capture", "router_logits")

        # top-k selection counts on the target tokens only
        target_ids = tok(answer, add_special_tokens=False).input_ids
        s, e = find_sub_list(target_ids, prompt_ids)[-1]
        sel = np.stack([topk_membership(logits[lid][s:e + 1])
                        for lid in sorted(logits)])
        cnt = sel.sum(axis=1)  # (layer, expert)
        counts[key] = cnt if counts[key] is None else counts[key] + cnt
        totals[key] += e - s + 1

print(f"detection tokens: digits={totals['digits']} "
      f"words={totals['words']}")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 21.27it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  5.51it/s, est. speed input: 198.35 toks/s, output: 5.51 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  5.51it/s, est. speed input: 198.35 toks/s, output: 5.51 toks/s]

...

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 1031.81it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 39.94it/s, est. speed input: 1001.29 toks/s, output: 40.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 37.86it/s, est. speed input: 1001.29 toks/s, output: 40.00 toks/s]

detection tokens: digits=38 words=38


## 3. Risk difference

`Δ(layer, expert) = p_digits − p_words`: experts with large positive Δ are
selected for digit tokens but not word tokens.

In [4]:
risk_diff = counts["digits"] / totals["digits"] \
    - counts["words"] / totals["words"]

flat = np.argsort(np.abs(risk_diff), axis=None)[::-1]
print("top behavior-linked experts (layer, expert, Δ):")
for idx in flat[:10]:
    layer, expert = divmod(int(idx), N_EXPERTS)
    print(f"  L{layer:02d} E{expert:02d}  "
          f"Δ={risk_diff[layer, expert]:+.2f}")

top behavior-linked experts (layer, expert, Δ):
  L03 E09  Δ=+0.50
  L03 E06  Δ=-0.39
  L14 E04  Δ=-0.34
  L15 E47  Δ=-0.34
  L15 E50  Δ=-0.32
  L05 E15  Δ=-0.29
  L03 E61  Δ=+0.29
  L14 E57  Δ=-0.26
  L01 E18  Δ=+0.26
  L02 E28  Δ=+0.26


## 4. Save the steering config

Deactivating the digit-linked experts (positive Δ) steers *away from
digits*. On OLMoE, 100 deactivated experts (~10% of 1024) flips greedy
counting to written number words — see `steermoe_steer.ipynb`.
(Fewer experts only perturb phrasing; many more degrade generation. The
paper tunes this count per model and task, Table A.2.)

In [5]:
N_DEACT = 100

deact = {}
taken = 0
for idx in flat:
    layer, expert = divmod(int(idx), N_EXPERTS)
    if risk_diff[layer, expert] <= 0:
        continue
    deact.setdefault(layer, []).append(expert)
    taken += 1
    if taken == N_DEACT:
        break

with open("steermoe_digits.json", "w") as f:
    json.dump({"layer_configs": {
        str(layer): {"mode": "deactivate", "expert_ids": ids}
        for layer, ids in deact.items()
    }}, f, indent=2)
print(f"saved steermoe_digits.json: {taken} experts "
      f"across {len(deact)} layers")

saved steermoe_digits.json: 100 experts across 16 layers
